### Clustering

- I need to do some clustering algos and see which does best
- Select which columns to use


What am I hoping clustering will do?
- Make more "like" groupings of school districts based on certain demographic data. 
- I believe the clusters will still have normal distribution curves, and linear relationships.

What columns are allowed in clustering -- For this one I want to make two selection types, one with ethnic data and one without as some grants or other funding does not like considering that information when benchmarking
- Eco-dis %
- student body size
- average teacher pay
- Ethnic homogeneity with students and staff
- Staff size


What makes up a row?
- a row will be made up of one school with all of its given metrics
- I want to only select schools who's metrics have not drastically changed in the past 3 years. What is why I have the historical data as I believe if a school has hd drastic changes they will be an outlier and throw off the model. for the school years data I will want to use the 2024 numbers as I believe any miss calculations have been adjusted by then (not sure if they even upload corrections but don't feel like looking as of now).
- I will need to normalize all the numbers down to aq 0 to 1 scale so large differences aren't causing issues.

I would also like to run an algo to see how strongly any one variable seems to be in relation to the overall outcomes

Right now, my data probably looks like a few loosely clouds semi clustered and overlapping because nothing is normalized and teacher pay of $40,000 and eco-dis count of 270 out of 3000 students are working together to make the a shape but are working on completely different scales, so I need an approach that with a very loss clustering algo until I clean further.

In [15]:
import pandas as pd
import sklearn


In [16]:
student_df = pd.read_csv(r"Clean Datasets\student_analysis_df.csv", dtype={'YearDistrict ID': str, 'DISTRICT': str, 'Year': str})
staff_df = pd.read_csv(r"Clean Datasets\staff_analysis_df.csv", dtype={'YearDistrict ID': str, 'DISTRICT': str, 'Year': str})


## Exploratory Clustering

- find rates for student and staff data
- Z-Score normalize
- 

- Stretch goal: Principal Components Analysis (PAC) on eco-dis, homeless


In [17]:
# Defining columns to keep
student_col_to_keep = ['YearDistrict ID', 'DISTRICT', 'Year', 'All Students Count', 'Special Ed Count',
                        'Bilingual/ESL Count', 'Career & Technical Education Count',
                        'Gifted & Talented Count', 'EB/EL Count', 'Econ Disadv Count',
                        'Non-Educationally Disadv Count', 'At Risk Count',
                        'American Indian Count', 'Asian Count', 'Pacific Islander Count',
                        'Two or More Races Count', 'African American Count', 'Hispanic Count',
                        'White Count', 'Male Count', 'Female Count', 'Dyslexia Count',
                        'Section 504 Count', 'Title I Count', 'Homeless Count',
                        'Immigrant Count', 'Migrant Count',
                        'Total Students with Disabilities Count',
                        'Intellectual Disabilities Count', 'Physical Disabilities Count',
                        'Behavioral Disabilities Count', 'Autism Count', 'District Name', 'County ID',
                        'County Name', 'Region', 'Charter Flag', 'DFLALTED']

staff_col_to_keep = [   'YearDistrict ID', 'Year', 'DISTRICT',       
                        'Teacher Total Full Time Equiv Count',
                        'Support Total Full Time Equiv Count',
                        'School Admin Total Full Time Equiv Count',
                        'Central Admin Total Full Time Equiv Count',
                        'Educ Aide Total Full Time Equiv Count',
                        'Teacher Total Base Salary Total', 'Support Total Base Salary Total',
                        'School Admin Total Base Salary Total',
                        'Teacher Beginning Full Time Equiv Count',
                        'Teacher 1-5 Years Full Time Equiv Count',
                        'Teacher 6-10 Years Full Time Equiv Count',
                        'Teacher 11-20 Years Full Time Equiv Count',
                        'Teacher 21-30 Years Full Time Equiv Count',
                        'Teacher > 30 Years Full Time Equiv Count',
                        'Teacher Beginning Base Salary Total',
                        'Teacher 1-5 Years Base Salary Total',
                        'Teacher 6-10 Years Base Salary Total',
                        'Teacher 11-20 Years Base Salary Total',
                        'Teacher 21-30 Years Base Salary Total',
                        'Teacher > 30 Years Base Salary Total',
                        'Teacher No Degree Full Time Equiv Count',
                        'Teacher BA Degree Full Time Equiv Count',
                        'Teacher MS Degree Full Time Equiv Count',
                        'Teacher PH Degree Full Time Equiv Count',
                        'Teacher American Indian Full Time Equiv Count',
                        'Teacher Pacific Islander Full Time Equiv Count',
                        'Teacher Asian Full Time Equiv Count',
                        'Teacher African American Full Time Equiv Count',
                        'Teacher Hispanic Full Time Equiv Count',
                        'Teacher White Full Time Equiv Count',
                        'Teacher Two or more races Full Time Equiv Count',
                        'Teacher Male Full Time Equiv Count',
                        'Teacher Female Full Time Equiv Count',
                        'Teacher Regular Program Full Time Equiv Count',
                        'Teacher Career & Technical Prgms Full Time Equiv Count',
                        'Teacher Bilingual Program Full Time Equiv Count',
                        'Teacher Compensatory Program Full Time Equiv Count',
                        'Teacher Gifted & Talented Program Full Time Equiv Count',
                        'Teacher Special Education Full Time Equiv Count',
                        'Teacher Other Full Time Equiv Count', 'Teacher Turnover Numerator',
                        'Teacher Turnover Denominator', 'Principal Experience Total',
                        'Principal Tenure Total', 'Assistant Principal Experience Total',
                        'Assistant Principal Tenure Total',
                        'Teacher Incentive Allotment Master Head Count',
                        'Teacher Incentive Allotment Exemplary Head Count',
                        'Teacher Incentive Allotment Recognized Head Count']

In [18]:
# create filtered dataframes
student_df_1 = student_df[student_col_to_keep]
staff_df_1 = staff_df[staff_col_to_keep]

# student_df_1
# staff_df_1

In [19]:
# Select Student Numerators
student_numerators = [ 'Special Ed Count', 'Bilingual/ESL Count',
                        'Career & Technical Education Count', 'Gifted & Talented Count',
                        'EB/EL Count', 'Econ Disadv Count', 'Non-Educationally Disadv Count',
                        'At Risk Count', 'American Indian Count', 'Asian Count',
                        'Pacific Islander Count', 'Two or More Races Count',
                        'African American Count', 'Hispanic Count', 'White Count', 'Male Count',
                        'Female Count', 'Dyslexia Count', 'Section 504 Count', 'Title I Count',
                        'Homeless Count', 'Immigrant Count', 'Migrant Count',
                        'Total Students with Disabilities Count',
                        'Intellectual Disabilities Count', 'Physical Disabilities Count',
                        'Behavioral Disabilities Count', 'Autism Count']

Explore shrinkage estimation/ Bayesian smoothing for proportions

In [20]:
# Function: Find rate of num/denom
def find_rate(df, numerator, denominator):
    """
    Creates a rate column for any given value
    df: DataFrame
    Numerator: top number of a fraction (the part)
    Denominator: bottom number of a fraction (the whole)
    """
    
    if type(numerator) == list:
        for i in numerator:
            df[i+"_Rate"] = df[i]/df[denominator]
    else:
        df[numerator+"_Rate"] = df[numerator]/df[denominator]

    return df

In [21]:
# Function: Calculating z score
def zscore(df: pd.DataFrame):
    """
    Calculates the z score of a column grouped by year
    df: DataFrame
    """
    for col in df.columns:
        if "_Rate" in col:
            z_col = col.replace("_Rate", "") + "_ZScore"
            df[z_col] = (
                df[col] -
                df.groupby('Year')[col].transform('mean')
            ) / df.groupby('Year')[col].transform('std')

    return df

In [22]:
student_df_2 = find_rate(student_df_1, student_numerators, 'All Students Count')
student_df_2 = zscore(student_df_2)

In [23]:
student_df_2

,YearDistrict ID,DISTRICT,Year,All Students Count,Special Ed Count,Bilingual/ESL Count,Career & Technical Education Count,Gifted & Talented Count,EB/EL Count,Econ Disadv Count,...,Section 504 Count_ZScore,Title I Count_ZScore,Homeless Count_ZScore,Immigrant Count_ZScore,Migrant Count_ZScore,Total Students with Disabilities Count_ZScore,Intellectual Disabilities Count_ZScore,Physical Disabilities Count_ZScore,Behavioral Disabilities Count_ZScore,Autism Count_ZScore
0,2017001902,001902,2017,576,82,3,150.0,55,3,192,...,NaN,NaN,NaN,NaN,NaN,1.772536,2.978282,0.452432,-0.393700,NaN
1,2017001903,001903,2017,1267,139,19,299.0,39,19,673,...,NaN,NaN,NaN,NaN,NaN,0.631982,0.729035,0.606462,-0.237393,NaN
2,2017001904,001904,2017,846,75,18,229.0,59,18,446,...,NaN,NaN,NaN,NaN,NaN,-0.103474,0.456307,-1.161736,-0.282013,NaN
3,2017001906,001906,2017,377,37,6,118.0,24,6,172,...,NaN,NaN,NaN,NaN,NaN,0.228033,1.881883,NaN,NaN,NaN
4,2017001907,001907,2017,3453,303,516,1222.0,93,557,2587,...,NaN,NaN,NaN,NaN,NaN,-0.135004,-0.342326,-0.503674,-0.042862,0.551144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10327,2025252902,252902,2025,225,42,4,73.0,3,4,140,...,0.442347,0.670510,-0.576063,-0.534356,-0.299411,0.539094,0.358486,-0.287510,0.480125,0.158835
10328,2025252903,252903,2025,689,145,98,202.0,27,98,383,...,0.744044,-0.245235,0.387659,1.607863,-0.299411,1.058976,-0.380101,2.893909,NaN,1.496989
10329,2025253901,253901,2025,3282,566,1072,940.0,233,981,2917,...,-0.128337,0.670510,1.885461,-0.144596,0.521116,0.228458,0.460316,-0.434492,-0.660263,0.769648
10330,2025254901,254901,2025,1635,298,40,575.0,127,41,1329,...,0.300226,0.670510,1.341721,-0.474173,0.194712,0.442834,0.118166,0.002104,0.051675,0.663247


In [24]:
staff_df_1.columns

Index(['YearDistrict ID', 'Year', 'DISTRICT',
       'Teacher Total Full Time Equiv Count',
       'Support Total Full Time Equiv Count',
       'School Admin Total Full Time Equiv Count',
       'Central Admin Total Full Time Equiv Count',
       'Educ Aide Total Full Time Equiv Count',
       'Teacher Total Base Salary Total', 'Support Total Base Salary Total',
       'School Admin Total Base Salary Total',
       'Teacher Beginning Full Time Equiv Count',
       'Teacher 1-5 Years Full Time Equiv Count',
       'Teacher 6-10 Years Full Time Equiv Count',
       'Teacher 11-20 Years Full Time Equiv Count',
       'Teacher 21-30 Years Full Time Equiv Count',
       'Teacher > 30 Years Full Time Equiv Count',
       'Teacher Beginning Base Salary Total',
       'Teacher 1-5 Years Base Salary Total',
       'Teacher 6-10 Years Base Salary Total',
       'Teacher 11-20 Years Base Salary Total',
       'Teacher 21-30 Years Base Salary Total',
       'Teacher > 30 Years Base Salary Total',

In [25]:
teacher_ep_numerator  =['Teacher Beginning Full Time Equiv Count',
                        'Teacher 1-5 Years Full Time Equiv Count',
                        'Teacher 6-10 Years Full Time Equiv Count',
                        'Teacher 11-20 Years Full Time Equiv Count',
                        'Teacher 21-30 Years Full Time Equiv Count',
                        'Teacher > 30 Years Full Time Equiv Count']
teacher_pay_numerator = ['Teacher Beginning Base Salary Total',
                        'Teacher 1-5 Years Base Salary Total',
                        'Teacher 6-10 Years Base Salary Total',
                        'Teacher 11-20 Years Base Salary Total',
                        'Teacher 21-30 Years Base Salary Total',]

teacher_edu_numerator = ['Teacher No Degree Full Time Equiv Count',
                        'Teacher BA Degree Full Time Equiv Count',
                        'Teacher MS Degree Full Time Equiv Count',
                        'Teacher PH Degree Full Time Equiv Count',]

teacher_eth_numerator = ['Teacher American Indian Full Time Equiv Count',
                        'Teacher Pacific Islander Full Time Equiv Count',
                        'Teacher Asian Full Time Equiv Count',
                        'Teacher African American Full Time Equiv Count',
                        'Teacher Hispanic Full Time Equiv Count',
                        'Teacher White Full Time Equiv Count',
                        'Teacher Two or more races Full Time Equiv Count',]



In [26]:
staff_df_2 = find_rate(staff_df_1, teacher_ep_numerator, 'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, teacher_edu_numerator,'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, teacher_eth_numerator,'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, 'Teacher Total Base Salary Total', 'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, teacher_pay_numerator, 'Teacher Total Base Salary Total')

staff_df_2 = zscore(staff_df_2)

In [27]:
staff_df_2

,YearDistrict ID,Year,DISTRICT,Teacher Total Full Time Equiv Count,Support Total Full Time Equiv Count,School Admin Total Full Time Equiv Count,Central Admin Total Full Time Equiv Count,Educ Aide Total Full Time Equiv Count,Teacher Total Base Salary Total,Support Total Base Salary Total,...,Teacher African American Full Time Equiv Count_ZScore,Teacher Hispanic Full Time Equiv Count_ZScore,Teacher White Full Time Equiv Count_ZScore,Teacher Two or more races Full Time Equiv Count_ZScore,Teacher Total Base Salary Total_ZScore,Teacher Beginning Base Salary Total_ZScore,Teacher 1-5 Years Base Salary Total_ZScore,Teacher 6-10 Years Base Salary Total_ZScore,Teacher 11-20 Years Base Salary Total_ZScore,Teacher 21-30 Years Base Salary Total_ZScore
0,2017001902,2017,001902,52.4,6.9,4.1,2.5,13.5,2418873.4,350003.3,...,-0.140774,-0.625037,0.694993,-0.568025,-0.155789,-0.614257,-1.291227,-0.961874,0.908384,NaN
1,2017001903,2017,001903,103.0,7.0,6.0,2.0,26.8,4628074.1,346087.0,...,-0.293504,-0.497197,0.653994,-0.568025,-0.403027,-0.645704,-0.946364,-0.651049,1.893220,NaN
2,2017001904,2017,001904,59.0,5.0,4.0,3.0,19.9,2580639.3,276976.0,...,0.100564,-0.634301,0.524849,0.780613,-0.643054,-0.554558,-0.358022,-0.206132,0.104762,NaN
3,2017001906,2017,001906,35.3,2.0,3.0,5.0,7.0,1548155.4,90675.0,...,0.009829,-0.707851,0.697270,-0.568025,-0.619427,-0.701072,0.003206,-0.643622,1.284956,NaN
4,2017001907,2017,001907,263.5,28.0,17.0,8.0,74.2,11827846.6,1459280.2,...,0.314959,-0.287903,0.096196,0.639863,-0.412139,-0.185424,0.276714,0.295957,-0.660803,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10327,2025252902,2025,252902,24.8,0.7,2.0,1.0,10.2,1479470.0,49629.0,...,-0.506599,-0.519679,0.784759,-0.660173,0.120995,-0.325390,-1.148565,1.058448,-0.741073,2.207499
10328,2025252903,2025,252903,73.3,7.7,6.3,1.0,1.0,4098574.3,515342.9,...,-0.311931,-0.525905,0.649487,-0.660173,-0.425595,-0.440485,0.074065,0.141244,-0.198378,-0.326495
10329,2025253901,2025,253901,240.2,47.4,14.3,11.0,62.7,13664248.8,3345159.2,...,-0.506599,3.130326,-2.395256,-0.307719,-0.283592,0.375624,-0.096654,-0.943204,0.571327,-0.030143
10330,2025254901,2025,254901,107.8,22.9,13.2,7.0,62.8,6129788.5,1547240.3,...,-0.440416,2.945535,-2.253190,-0.660173,-0.287152,0.226400,0.117066,-0.194242,0.028111,-0.083855


In [28]:
# Select Teacher Numerators

